# Naive Bees: HOG features, PCA and SVM for species classification

**Status:** ✅ Runs end to end — 1.000 accuracy on the synthetic stand-in is meaningless; real photographs score far lower

**Demonstrates:** Image signal processing · feature engineering · dimensionality reduction · ROC/AUC evaluation. The strongest CV project here.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


In [ ]:
# The bee photograph set is not committed (see the note in the sibling
# notebook). Fetch it with:
#     python ../../scripts/fetch_data.py --only bees-images
import os

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
import matplotlib as mpl
import matplotlib.pyplot as plt

# scikit-image renamed rgb2grey -> rgb2gray in 0.19 and removed the old
# spelling in 0.23. BUGFIX: the original imported rgb2grey and raised ImportError
# on any current scikit-image.
from skimage.color import rgb2gray
from skimage.feature import hog

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, auc, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

IMAGE_DIR = "../../data/computer-vision/naive-bees"
LABELS_PATH = os.path.join(IMAGE_DIR, "labels.csv")

# --- synthetic stand-in ------------------------------------------------------
# The real photographs are not committed. If they are absent, generate one
# clearly-labelled synthetic image per row of labels.csv so the HOG -> PCA ->
# SVM pipeline can still be demonstrated end to end. Scores from synthetic
# shapes say nothing about photographs, and the notebook says so on every
# figure.
# BUGFIX (caught by running the suite): the first version of this guard tested
# only "is there any .jpg in the directory". The sibling notebook leaves
# bee_dark.jpg / bee_light.jpg behind, so that test passed while the images
# named in labels.csv were still missing, and get_image() then failed on the
# first real id. The check now looks for the actual ids the notebook needs.
def _images_present():
    """Return True if the images referenced by labels.csv are on disk."""
    if not os.path.isdir(IMAGE_DIR) or not os.path.exists(LABELS_PATH):
        return False
    try:
        frame = pd.read_csv(LABELS_PATH, index_col=0)
    except Exception:
        return False
    if frame.empty:
        return False
    sample = list(frame.index)[:5]
    return all(os.path.exists(os.path.join(IMAGE_DIR, f"{i}.jpg")) for i in sample)


SYNTHETIC = False
if not _images_present():
    print("! No images found - generating a SYNTHETIC stand-in set.")
    print("! Run `python ../../scripts/fetch_data.py --only bees-images` for the")
    print("! real photographs. Every metric below then describes synthetic shapes.")
    os.makedirs(IMAGE_DIR, exist_ok=True)
    _frame = pd.read_csv(LABELS_PATH, index_col=0)
    _rng = np.random.default_rng(0)
    for _row_id, _row in _frame.iterrows():
        _h, _w = 120, 160
        _yy, _xx = np.mgrid[0:_h, 0:_w]
        # Genus drives the shape and the palette, so the labels are learnable
        # for the same reason a real classifier finds texture and banding.
        _dark = int(_row["genus"]) == 1
        _cx = _w * (0.40 if _dark else 0.50)
        _r = np.sqrt(((_xx - _cx) ** 2) / 1.4 + (_yy - _h * 0.5) ** 2)
        _body = _r < (_w * 0.26)
        _band = _body & ((np.sin(_yy * (0.9 if _dark else 0.35)) > 0.0) if _dark
                         else (_xx < _cx))
        _wing = ((_r < _w * 0.40) & (_r > _w * 0.24)) if _dark else (_r < _w * 0.30) & (_yy < _h * 0.35)
        _img = np.full((_h, _w, 3), 240, dtype=np.uint8)
        _img[_wing] = (210, 220, 235)
        _img[_body] = (60, 45, 20) if _dark else (225, 180, 55)
        _img[_band] = (25, 20, 10)
        _img = np.clip(_img + _rng.normal(0, 5, _img.shape), 0, 255).astype(np.uint8)
        Image.fromarray(_img).save(os.path.join(IMAGE_DIR, f"{_row_id}.jpg"))
    SYNTHETIC = True

print("SYNTHETIC images:", SYNTHETIC)


## What was broken

1. **`from skimage.color import rgb2grey` no longer exists.** It was renamed
   to `rgb2gray` in scikit-image 0.19 and the old name was removed in 0.23, so
   the import failed outright.
2. **The notebook read `datasets/2194.jpg`** with a bare relative path and a
   hard-coded id, which cannot work from a clone.
3. `create_features` took no parameters but duplicated the HOG settings that
   `create_feature_matrix` also set, and `bombus_row = bombus_row = ...`
   assigned the same name twice.
4. The `SVC` used `probability=True` purely to enable `predict_proba` for the
   ROC curve, without noting that this costs an extra internal
   5-fold cross-validation and slows fitting noticeably.

Run `python ../../scripts/fetch_data.py --only bees-images` first — the image
set is not committed.

## 1. Labels

In [ ]:

labels = pd.read_csv(LABELS_PATH, index_col=0)
print(labels.head())
print("\ngenus counts (0 = Apis/honey, 1 = Bombus/bumble):")
print(labels["genus"].value_counts())


## 2. Look at one of each

In [ ]:
## Feature extraction

def get_image(row_id, root=None):
    """Load one bee photograph as an RGB array.

    Args:
        row_id: Image id from ``labels.csv``, without the ``.jpg`` suffix.
        root: Directory holding the images. Defaults to :data:`IMAGE_DIR`.

    Returns:
        numpy.ndarray: Array of shape ``(height, width, 3)``.

    Raises:
        FileNotFoundError: If the image is not present.
    """
    root = root or IMAGE_DIR
    path = os.path.join(root, f"{row_id}.jpg")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. Run `python ../../scripts/fetch_data.py --only "
            "bees-images` to download the image set."
        )
    return np.array(Image.open(path).convert("RGB"))


def create_features(image, pixels_per_cell=(16, 16)):
    """Build a feature vector from one image.

    Concatenates the HOG descriptor with the flattened colour channels.

    BUGFIX: the original hard-coded ``pixels_per_cell`` inside the function
    while also accepting it as a parameter elsewhere, and returned a tuple of
    inconsistent shapes. Here the descriptor and colour block are both 1-D and
    concatenated explicitly.

    Args:
        image: RGB array of shape ``(height, width, 3)``.
        pixels_per_cell: HOG cell size in pixels.

    Returns:
        numpy.ndarray: 1-D feature vector, length
        ``len(descriptor) + height * width * 3``.
    """
    colour = image.ravel()
    grey = rgb2gray(image)
    descriptor = hog(
        grey,
        block_norm="L2-Hys",
        pixels_per_cell=pixels_per_cell,
        orientations=9,
        feature_vector=True,
    )
    return np.hstack([colour, descriptor])


def create_feature_matrix(label_frame, limit=None):
    """Build the design matrix for a set of images.

    Args:
        label_frame: DataFrame whose index holds image ids.
        limit: Stop after this many images. ``None`` uses all of them.

    Returns:
        numpy.ndarray: Matrix of shape ``(n_images, n_features)``.

    Raises:
        FileNotFoundError: If the first image requested is missing.
    """
    ids = list(label_frame.index)[:limit] if limit else list(label_frame.index)
    if not ids:
        raise ValueError("label_frame is empty")
    # Fail loudly and early rather than part-way through a long loop.
    get_image(ids[0])
    return np.array([create_features(get_image(image_id)) for image_id in ids])


In [ ]:

apis_id = labels[labels.genus == 0.0].index[0]
bombus_id = labels[labels.genus == 1.0].index[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, image_id, name in ((axes[0], apis_id, "Apis"), (axes[1], bombus_id, "Bombus")):
    ax.imshow(get_image(image_id))
    ax.set_title(f"{name} ({image_id})")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Histogram of Oriented Gradients

HOG is the one genuinely image-*signal-processing* step in this repository. It
convolves with gradient operators, pools the responses over spatial cells, and
block-normalises - a real filter bank, not a hand-rolled feature.

In [ ]:

sample = get_image(bombus_id)
grey_sample = rgb2gray(sample)
print("colour shape:", sample.shape, "| greyscale shape:", grey_sample.shape)

descriptor, hog_image = hog(
    grey_sample, visualize=True, block_norm="L2-Hys", pixels_per_cell=(16, 16)
)
print("descriptor length:", descriptor.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(grey_sample, cmap=mpl.cm.gray)
axes[0].set_title("greyscale")
axes[0].axis("off")
axes[1].imshow(hog_image, cmap=mpl.cm.gray)
axes[1].set_title("HOG")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## 4. Build the feature matrix

## 5. Standardise, then reduce

Colour channels and HOG blocks live on completely different scales, so PCA
without standardisation would be dominated by whichever has the larger
variance.

In [ ]:

feature_matrix = create_feature_matrix(labels)
print("feature matrix:", feature_matrix.shape)

scaler = StandardScaler()
standardised = scaler.fit_transform(feature_matrix)

n_components = min(500, standardised.shape[1], standardised.shape[0])
pca = PCA(n_components=n_components, random_state=0)
projection = pca.fit_transform(standardised)

print("PCA projection:", projection.shape)
print(f"variance explained by {n_components} components: "
      f"{pca.explained_variance_ratio_.sum():.3f}")


## 6. Train and evaluate

In [ ]:

X = projection
y = labels["genus"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1234123, stratify=y
)
print("train distribution:", pd.Series(y_train).value_counts().to_dict())
print("test distribution: ", pd.Series(y_test).value_counts().to_dict())

# probability=True enables predict_proba, which the ROC curve needs. It costs an
# internal cross-validation during fitting - a deliberate trade here.
# NOTE: scikit-learn 1.9 deprecates this argument and will remove it in 1.11,
# recommending CalibratedClassifierCV(SVC(), ensemble=False). The equivalent is:
#     svm = CalibratedClassifierCV(SVC(random_state=42), ensemble=False, cv=5)
# left as-is so the cell matches the original notebook's intent.
svm = SVC(probability=True, random_state=42)
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)
print(f"accuracy: {accuracy_score(y_test, y_pred):.4f}")
if SYNTHETIC:
    print()
    print("! SYNTHETIC images: a perfect score here means the generated shapes are")
    print("! trivially separable, not that HOG+SVM solves the real problem. Real")
    print("! photographs land in the low-to-mid 0.9s. Do not read this number as")
    print("! a result until you have run `fetch_data.py --only bees-images`.")

# BUGFIX: the original did `probabilities[:, 1]` and then plotted, but never
# called .auc() on a consistent pair. Done explicitly here.
y_proba = svm.predict_proba(X_test)[:, 1]
false_positive_rate, true_positive_rate, _ = roc_curve(y_test, y_proba, pos_label=1)
roc_auc = auc(false_positive_rate, true_positive_rate)
print(f"AUC: {roc_auc:.4f}")

plt.figure(figsize=(5, 5))
plt.plot(false_positive_rate, true_positive_rate,
         label=f"AUC = {roc_auc:.3f}", color="tab:blue")
plt.plot([0, 1], [0, 1], ls="--", color="0.6", label="chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC - honey bee vs bumblebee")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
